In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
df = pd.read_csv('cardekho_imputated.csv')

In [3]:
df.drop(columns=['Unnamed: 0','car_name'], axis=1, inplace=True)

In [4]:
df.head()

,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [5]:
df['brand'].unique()

array(['Maruti', 'Hyundai', 'Ford', 'Renault', 'Mini', 'Mercedes-Benz',
       'Toyota', 'Volkswagen', 'Honda', 'Mahindra', 'Datsun', 'Tata',
       'Kia', 'BMW', 'Audi', 'Land Rover', 'Jaguar', 'MG', 'Isuzu',
       'Porsche', 'Skoda', 'Volvo', 'Lexus', 'Jeep', 'Maserati',
       'Bentley', 'Nissan', 'ISUZU', 'Ferrari', 'Mercedes-AMG',
       'Rolls-Royce', 'Force'], dtype=object)

In [6]:
len(df['model'].unique())

120

In [7]:
df['brand'] = df['brand'].str.strip().str.title()

In [8]:
x = df[[feature for feature in df.columns if feature != "selling_price"]]
y = df['selling_price']

# Train Test Split

In [9]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

### Encoding --

In [10]:
categorical_feature = x.select_dtypes(include='object').columns
numerical_features = x.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

oh_encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
std_scaler = StandardScaler()

processor = ColumnTransformer([
    ("OneHotEncoder", oh_encoder, categorical_feature),
    ("StandardScaler", std_scaler, numerical_features)
])

In [11]:
x_train = processor.fit_transform(x_train)
x_test = processor.transform(x_test)

C:\Users\anupa\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


# Different Model training--

In [12]:
def metric_eval(true, pred):
    mae = mean_absolute_error(true, pred)
    mse = mean_squared_error(true, pred)
    r2 = r2_score(true, pred)
    rmse = np.sqrt(mse)
    return mae, mse, rmse, r2

In [15]:
pip install xgboost


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 3.2 MB/s eta 0:00:16
   - -------------------------------------- 1.8/48.9 MB 3.4 MB/s eta 0:00:14
   -- ------------------------------------- 3.1/48.9 MB 4.2 MB/s eta 0:00:11
   --- ------------------------------------ 4.5/48.9 MB 4.8 MB/s eta 0:00:10
   ---- ----------------------------------- 6.0/48.9 MB 5.3 MB/s eta 0:00:09
   ----- ---------------------------------- 7.3/48.9 MB 5.6 MB/s eta 0:00:08
   ------- -------------------------------- 9.2/48.9 MB 5.9 MB/s eta 0:00:07
   -------- ------------------------------- 10.7/48.9 MB 6.1 MB/s eta 0:00:07
   --------- ------------------------------ 12.1/48.9 MB 6.1 MB/s eta 0:00:07
   ----------- ---------------------------- 13.6/48.9 MB 6.3 MB/s eta 0:00:06
   ------------ --------------------------- 15.5/48.9 MB 6.5 MB/s eta 0:00:06
   -----

In [16]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge" : Ridge(),
    "Lasso" : Lasso(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest" : RandomForestRegressor(),
    "Ada Boost" : AdaBoostRegressor(),
    "Gradient Boost" : GradientBoostingRegressor(),
    "XGBoost Regressor" : XGBRegressor()
}

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)

    tr_mae, tr_mse, tr_rmse, tr_r2 =metric_eval(y_train, train_pred)
    t_mae, t_mse, t_rmse, t_r2 =metric_eval(y_test, test_pred)

    print(f"For {list(models.keys())[i]} model -----\n")
    print('Model performance for Training set')
    print("- Mean Squared Error: {:.4f}".format(tr_mse))
    print("- Root Mean Squared Error: {:.4f}".format(tr_rmse))
    print("- Mean Absolute Error: {:.4f}".format(tr_mae))
    print("- R2 Score: {:.4f}".format(tr_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Mean Squared Error: {:.4f}".format(t_mse))
    print("- Root Mean Squared Error: {:.4f}".format(t_rmse))
    print("- Mean Absolute Error: {:.4f}".format(t_mae))
    print("- R2 Score: {:.4f}".format(t_r2))
    
    print('='*35)
    print('\n')
    

For Linear Regression model -----

Model performance for Training set
- Mean Squared Error: 111927776457.1274
- Root Mean Squared Error: 334556.0887
- Mean Absolute Error: 164483.3289
- R2 Score: 0.8612
----------------------------------
Model performance for Test set
- Mean Squared Error: 171203460353.7639
- Root Mean Squared Error: 413767.3989
- Mean Absolute Error: 181546.1412
- R2 Score: 0.7799


For Ridge model -----

Model performance for Training set
- Mean Squared Error: 129778203994.8247
- Root Mean Squared Error: 360247.4205
- Mean Absolute Error: 175709.7039
- R2 Score: 0.8391
----------------------------------
Model performance for Test set
- Mean Squared Error: 180115732071.6005
- Root Mean Squared Error: 424400.4383
- Mean Absolute Error: 192198.8755
- R2 Score: 0.7684




C:\Users\anupa\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.507e+14, tolerance: 9.322e+11
  model = cd_fast.enet_coordinate_descent(


For Lasso model -----

Model performance for Training set
- Mean Squared Error: 111928214047.9011
- Root Mean Squared Error: 334556.7426
- Mean Absolute Error: 164492.3434
- R2 Score: 0.8612
----------------------------------
Model performance for Test set
- Mean Squared Error: 173666854470.7046
- Root Mean Squared Error: 416733.5533
- Mean Absolute Error: 182192.7574
- R2 Score: 0.7767


For Decision Tree model -----

Model performance for Training set
- Mean Squared Error: 426210181.9807
- Root Mean Squared Error: 20644.8585
- Mean Absolute Error: 4991.7517
- R2 Score: 0.9995
----------------------------------
Model performance for Test set
- Mean Squared Error: 79569012059.0197
- Root Mean Squared Error: 282079.7973
- Mean Absolute Error: 124616.5520
- R2 Score: 0.8977


For Random Forest model -----

Model performance for Training set
- Mean Squared Error: 15849977248.2438
- Root Mean Squared Error: 125896.6928
- Mean Absolute Error: 38746.5089
- R2 Score: 0.9803
------------------

# Hyperparameter tuning

In [23]:
rf_params = {
    'criterion' : ['squared_error', 'absolute_error', 'poisson'],
    'max_depth' : [40, 60, 80, 100, 120],
    'min_samples_split' : [2, 3, 4, 8, 10],
    'min_samples_leaf': [1, 2, 5, 10]
}

xgboost_params = {
    'n_estimators' : [100, 300, 500, 800, 1000],
    'learning_rate' : [0.01, 0.05, 0.1, 0.5],
    'max_depth' : [3, 5, 7, 10],
    'min_child_weight' : [1, 3, 8, 10],
    'subsample' : [0.2, 0.4, 0.6, 0.8, 1],
    'colsample_bytree' : [0.2, 0.4, 0.6, 0.8, 1],
    'reg_alpha' : [0, 0.1, 0.5, 1, 5],
    'reg_lambda' : [0, 0.1, 0.5, 1, 5]
}

In [25]:
from sklearn.model_selection import RandomizedSearchCV

models_best_params = {}

models = [
    ("Random Forest", RandomForestRegressor(), rf_params),
    ("XGBoost", XGBRegressor(), xgboost_params)
]

for name, model, params in models:
    randCv = RandomizedSearchCV(estimator=model,
                      param_distributions=params,
                      cv=3,
                      n_jobs=-1,
                      n_iter=30,
                      random_state=42,
                      scoring='r2')
    randCv.fit(x_train, y_train)
    models_best_params[name] = [randCv.best_params_, randCv.best_score_]
    
for val in models_best_params:
    print(models_best_params[val][0])
    print(models_best_params[val][1])
    

{'min_samples_split': 3, 'min_samples_leaf': 1, 'max_depth': 60, 'criterion': 'poisson'}
0.8493894504585168
{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 5, 'n_estimators': 1000, 'min_child_weight': 8, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
0.8677124977111816


# Tuning Model comaparision

In [26]:
models = {
    "Random Forest" : RandomForestRegressor(min_samples_split= 8, min_samples_leaf= 1, max_depth= 40, criterion= 'poisson'),
    "XGBoost" : XGBRegressor(subsample= 0.6, reg_lambda= 1, reg_alpha= 5, n_estimators= 1000, min_child_weight= 8, max_depth= 7, learning_rate= 0.1, colsample_bytree= 0.8)
}

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)

    tr_mae, tr_mse, tr_rmse, tr_r2 =metric_eval(y_train, train_pred)
    t_mae, t_mse, t_rmse, t_r2 =metric_eval(y_test, test_pred)

    print(f"For {list(models.keys())[i]} model -----\n")
    print('Model performance for Training set')
    print("- Mean Squared Error: {:.4f}".format(tr_mse))
    print("- Root Mean Squared Error: {:.4f}".format(tr_rmse))
    print("- Mean Absolute Error: {:.4f}".format(tr_mae))
    print("- R2 Score: {:.4f}".format(tr_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Mean Squared Error: {:.4f}".format(t_mse))
    print("- Root Mean Squared Error: {:.4f}".format(t_rmse))
    print("- Mean Absolute Error: {:.4f}".format(t_mae))
    print("- R2 Score: {:.4f}".format(t_r2))
    
    print('='*35)
    print('\n')
    

For Random Forest model -----

Model performance for Training set
- Mean Squared Error: 40805181058.5796
- Root Mean Squared Error: 202002.9234
- Mean Absolute Error: 60968.2914
- R2 Score: 0.9494
----------------------------------
Model performance for Test set
- Mean Squared Error: 57977424203.6787
- Root Mean Squared Error: 240785.0166
- Mean Absolute Error: 103568.9183
- R2 Score: 0.9255


For XGBoost model -----

Model performance for Training set
- Mean Squared Error: 9338777600.0000
- Root Mean Squared Error: 96637.3510
- Mean Absolute Error: 66969.8281
- R2 Score: 0.9884
----------------------------------
Model performance for Test set
- Mean Squared Error: 85130084352.0000
- Root Mean Squared Error: 291770.6023
- Mean Absolute Error: 104247.7266
- R2 Score: 0.8905


